# PD Voice Pipeline on Google Colab

This notebook clones the repository, installs dependencies, prepares data, and runs the pipeline.

In [ ]:
# 1) Clone repository and enter repo root
from pathlib import Path
import os
REPO_URL = "https://github.com/Nishi-BS23/PD-Pipeline.git"
REPO_DIR = "Pipeline-Implementation"
BRANCH = "master"

!rm -rf "$REPO_DIR"
!git clone -b "$BRANCH" --single-branch "$REPO_URL" "$REPO_DIR"

candidates = []
for p in Path('/content').rglob('audio_segmentation.py'):
    root = p.parent
    if (root / 'comparative_analysis.py').exists():
        candidates.append(root)
if not candidates:
    raise FileNotFoundError('Could not find repo root under /content')
REPO_ROOT = sorted(candidates, key=lambda x: len(str(x)))[0]
os.environ['PD_REPO_ROOT'] = str(REPO_ROOT)
%cd {REPO_ROOT}
print('Repo root:', REPO_ROOT)
print('Branch:', BRANCH)
!ls -la | sed -n '1,200p'

In [ ]:
# 2) Install dependencies
from pathlib import Path
import os
import torch

repo_env = os.environ.get('PD_REPO_ROOT', '')
if repo_env:
    REPO_ROOT = Path(repo_env)
else:
    candidates = []
    for p in Path('/content').rglob('audio_segmentation.py'):
        root = p.parent
        if (root / 'comparative_analysis.py').exists():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError('Could not find repo root under /content')
    REPO_ROOT = sorted(candidates, key=lambda x: len(str(x)))[0]
%cd {REPO_ROOT}
req_candidates = [
    Path('Wav2Vec2/requirements.txt'),
    Path('wav2vec2/requirements.txt'),
    Path('requirements.txt'),
]
req_path = next((p for p in req_candidates if p.exists()), None)
if req_path is None:
    print('requirements.txt not found; using direct package list')

!pip -q install --upgrade pip
if req_path is not None:
    print('Using requirements:', req_path)
    !pip -q install -r {req_path} openpyxl umap-learn
else:
    !pip -q install torch torchaudio transformers scipy scikit-learn pandas numpy matplotlib tqdm soundfile openpyxl umap-learn

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3) Provide data files
Place these inside your Google Drive folder (example: `MyDrive/PD_DATA`):
- `final_selected.xlsx`
- `mpower_voice_data_flac...` folder (can be `mpower_voice_data_flac` or `mpower_voice_data_flac-...`)

Then run the next cell to mount Drive and link/copy them into the repo root.

In [ ]:
# 4) Mount Drive and prepare dataset
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import glob
import shutil
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/PD_DATA')
repo_env = os.environ.get('PD_REPO_ROOT', '')
if repo_env:
    REPO_ROOT = Path(repo_env)
else:
    candidates = []
    for p in Path('/content').rglob('audio_segmentation.py'):
        root = p.parent
        if (root / 'comparative_analysis.py').exists():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError('Could not find repo root under /content')
    REPO_ROOT = sorted(candidates, key=lambda x: len(str(x)))[0]

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f'DRIVE_ROOT not found: {DRIVE_ROOT}')
if not REPO_ROOT.exists():
    raise FileNotFoundError(f'REPO_ROOT not found: {REPO_ROOT}')

meta_src = DRIVE_ROOT / 'final_selected.xlsx'
if not meta_src.exists():
    raise FileNotFoundError('final_selected.xlsx not found in DRIVE_ROOT')

raw_candidates = sorted(glob.glob(str(DRIVE_ROOT / 'mpower_voice_data_flac*')))
if not raw_candidates:
    raise FileNotFoundError('Could not find mpower_voice_data_flac* folder under DRIVE_ROOT')
raw_src = Path(raw_candidates[0])
try:
    inner = raw_src / 'mpower_voice_data_flac'
    raw_actual = inner if inner.is_dir() else raw_src
except OSError:
    raw_actual = raw_src

dst_meta = REPO_ROOT / 'final_selected.xlsx'
shutil.copy2(meta_src, dst_meta)
print('Copied metadata:', meta_src, '->', dst_meta)

dst_raw = REPO_ROOT / 'mpower_voice_data_flac'
if dst_raw.exists() or dst_raw.is_symlink():
    if dst_raw.is_symlink() or dst_raw.is_file():
        dst_raw.unlink()
    else:
        shutil.rmtree(dst_raw)
os.symlink(raw_actual, dst_raw, target_is_directory=True)
print('Linked raw folder:', dst_raw, '->', raw_actual)

%cd {REPO_ROOT}
!ls -la | sed -n '1,120p'

In [ ]:
# 5) Run pipeline (choose mode: 'full' or 'segment')
import torch
import os
from pathlib import Path

MODE = 'full'
MAX_PER_CLASS = 1000
SEED = 42
N_TRIALS = 20
FINAL_EPOCHS = 100

repo_env = os.environ.get('PD_REPO_ROOT', '')
if repo_env:
    REPO_ROOT = Path(repo_env)
else:
    candidates = []
    for p in Path('/content').rglob('audio_segmentation.py'):
        root = p.parent
        if (root / 'comparative_analysis.py').exists():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError('Could not find repo root under /content')
    REPO_ROOT = sorted(candidates, key=lambda x: len(str(x)))[0]
%cd {REPO_ROOT}
print('Running from:', REPO_ROOT)
print('CUDA available:', torch.cuda.is_available())
GPU_ARG = '--gpu 0' if torch.cuda.is_available() else ''
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

!python audio_segmentation.py --mode $MODE --raw-glob mpower_voice_data_flac --max-per-class $MAX_PER_CLASS --seed $SEED
!python Wav2Vec2/pipeline.py --mode $MODE $GPU_ARG --max-per-class $MAX_PER_CLASS --seed $SEED
!python HuBERT/pipeline.py --mode $MODE $GPU_ARG --max-per-class $MAX_PER_CLASS --seed $SEED
!python comparative_analysis.py --mode $MODE --n-trials $N_TRIALS --final-epochs $FINAL_EPOCHS --max-per-class $MAX_PER_CLASS --seed $SEED

In [ ]:
# 6) Show output artifacts
!find results -maxdepth 4 -type f | head -n 100

In [ ]:
# 7) (Optional) Save outputs to Drive
# !cp -r results /content/drive/MyDrive/PD_DATA/